# Modul A · Kapitel 1.3 — Logistische Regression

## Challenge: Ist dieser Tumor gutartig oder bösartig?

**Lernziel:** Du kannst das Verfahren aus Kapitel 1.2 auf eine **Klassifikation** übertragen —
und du kannst die Wahrscheinlichkeiten und den Schwellenwert deines Modells interpretieren.

Das Vorgehen kennst du schon: Modell, Kostenfunktion, trainieren, anwenden. Zwei Bausteine
werden ausgetauscht — und nur diese zwei musst du dazulernen.

> ⚠️ **Wichtiger Hinweis:** Die Daten in diesem Notebook sind **vollständig synthetisch** —
> am Computer erzeugt, keine echten Patientendaten. In der Medizin lässt sich Bösartigkeit
> selbstverständlich **nicht** allein aus der Tumorgröße ableiten; dafür braucht es Bildgebung,
> Gewebeproben und Ärztinnen und Ärzte. Wir benutzen das Beispiel, weil es mit **einem**
> Merkmal auskommt und weil hier besonders deutlich wird, warum ein Modell
> Wahrscheinlichkeiten ausgeben sollte statt eines schlichten Ja/Nein.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |
| 💬 | Diskussionsfrage — kurz überlegen, gerne mit der Nachbarin / dem Nachbarn |

**Wichtig:** Führe die Zellen **von oben nach unten** aus. Spätere Zellen brauchen die
Funktionen, die du vorher schreibst.

Es sind **6 Challenges**.

---
## 0 · Setup

▶️ Führe diese zwei Zellen aus. Danach sind alle Werkzeuge geladen und die Daten liegen bereit.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})
np.set_printoptions(suppress=True, precision=2)

print("Setup fertig ✔")

In [ ]:
def lade_daten(dateiname):
    """Liest eine der Workshop-CSVs ein — lokal oder in Google Colab."""
    kandidaten = [
        Path(dateiname),
        Path("data") / dateiname,
        Path("..") / "data" / dateiname,
        Path("challenges/logistische-regression/data") / dateiname,
    ]
    for pfad in kandidaten:
        if pfad.exists():
            print(f"Gelesen: {pfad}")
            return pd.read_csv(pfad)

    # Google Colab: Datei von Hand hochladen
    try:
        from google.colab import files
        print(f"'{dateiname}' nicht gefunden — bitte jetzt hochladen:")
        files.upload()
        return pd.read_csv(dateiname)
    except ImportError:
        raise FileNotFoundError(
            f"'{dateiname}' nicht gefunden. Lege die CSV neben dieses Notebook."
        )


df = lade_daten("tumor_daten.csv")
df.head()

---
## 1 · Eine andere Art von Frage

📖 In Kapitel 1.2 lautete die Frage: **„Wie teuer ist dieses Haus?“** Die Antwort war eine Zahl,
das war eine **Regression**.

Jetzt lautet sie: **„Ist dieser Tumor bösartig — ja oder nein?“** Die Antwort ist eine von zwei
Kategorien. Das ist eine **Klassifikation**, genauer: eine **binäre Klassifikation** (binär =
zwei mögliche Antworten).

Solche Fragen sind überall:

| Frage | 1 | 0 |
|---|---|---|
| Ist dieser Tumor bösartig? | bösartig | gutartig |
| Ist diese Mail Spam? | Spam | kein Spam |
| Kündigt dieser Kunde bald? | kündigt | bleibt |

Unser Datensatz hat genau zwei Spalten — mehr braucht es nicht:

| Spalte | Bedeutung |
|---|---|
| `tumor_groesse_mm` | Größe des Tumors in Millimetern |
| `boesartig` | **1** = bösartig, **0** = gutartig |

Diese 0/1-Spalte heißt **Label** — die richtige Antwort, aus der das Modell lernen soll.

In [ ]:
print(f"Patientinnen und Patienten: {len(df)}")
print(f"Spalten: {list(df.columns)}")
df.describe()

### 🛠️ Challenge 1 — Verschaff dir einen Überblick

Bevor man rechnet, schaut man sich die Daten an. Vier Zahlen sollen es sein:

1. wie viele Tumoren insgesamt **bösartig** sind,
2. der **Anteil** bösartiger Tumoren insgesamt,
3. der Anteil bösartiger Tumoren unter den **kleinen** (unter 15 mm),
4. der Anteil bösartiger Tumoren unter den **großen** (über 35 mm).

*Tipp: Weil die Spalte nur aus 0 und 1 besteht, ist ihr **Mittelwert** genau der Anteil an
Einsen. `df["boesartig"].mean()` reicht also für den Anteil.*

*Tipp zu 3 und 4: Mit `df.loc[bedingung, "boesartig"]` bekommst du nur die Zeilen, auf die
die Bedingung zutrifft — zum Beispiel `df["tumor_groesse_mm"] < 15`.*

In [ ]:
anzahl_boesartig = ...
anteil_boesartig = ...

anteil_kleine = ...   # Tumoren unter 15 mm
anteil_grosse = ...   # Tumoren über 35 mm

print(f"Bösartig insgesamt: {anzahl_boesartig} von {len(df)} ({anteil_boesartig:.1%})")
print(f"unter 15 mm:        {anteil_kleine:.1%} bösartig")
print(f"über 35 mm:         {anteil_grosse:.1%} bösartig")

In [ ]:
# ✅ Selbsttest
assert int(anzahl_boesartig) == 142, "Erwartet werden 142 bösartige Tumoren"
assert np.isclose(anteil_boesartig, 0.4733, atol=1e-3)
assert np.isclose(anteil_kleine, 0.0851, atol=1e-3), "Unter 15 mm sind es rund 8,5 %"
assert np.isclose(anteil_grosse, 0.9737, atol=1e-3), "Über 35 mm sind es rund 97,4 %"
print("✅ Challenge 1 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
anzahl_boesartig = df["boesartig"].sum()
anteil_boesartig = df["boesartig"].mean()

anteil_kleine = df.loc[df["tumor_groesse_mm"] < 15, "boesartig"].mean()
anteil_grosse = df.loc[df["tumor_groesse_mm"] > 35, "boesartig"].mean()

print(f"Bösartig insgesamt: {anzahl_boesartig} von {len(df)} ({anteil_boesartig:.1%})")
print(f"unter 15 mm:        {anteil_kleine:.1%} bösartig")
print(f"über 35 mm:         {anteil_grosse:.1%} bösartig")
```

**8,5 % gegen 97,4 %** — die Größe sagt offensichtlich etwas aus. Aber achte auf die Formulierung: *häufiger* bösartig, nicht *immer*. Unter den kleinen Tumoren sind welche bösartig, unter den großen welche gutartig. Genau darum geht es im nächsten Abschnitt.

</details>

---
## 2 · Erst schauen, dann rechnen

📖 Das Streudiagramm sieht diesmal ungewohnt aus: Auf der y-Achse stehen nur **zwei** Werte,
0 und 1. Alle Punkte liegen also auf zwei Linien.

Trotzdem — oder gerade deshalb — verrät es das Entscheidende: Wo endet die 0-Linie, wo beginnt
die 1-Linie, und **wie breit ist der Bereich, in dem beide gleichzeitig vorkommen?**

### 🛠️ Challenge 2 — Das Streudiagramm

Zeichne ein Streudiagramm mit **`tumor_groesse_mm`** auf der x-Achse und **`boesartig`** auf der
y-Achse.

*Tipp: `ax.scatter(x, y, s=30, alpha=0.35, color=BLAU, linewidths=0)`. Das `alpha` ist wichtig —
wo viele Punkte übereinanderliegen, wird es dadurch dunkler.*

*Tipp für die Beschriftung: `ax.set_yticks([0, 1])` und
`ax.set_yticklabels(["gutartig", "bösartig"])`.*

In [ ]:
fig, ax = plt.subplots()

...

plt.show()

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
fig, ax = plt.subplots()

ax.scatter(df["tumor_groesse_mm"], df["boesartig"], s=30, alpha=0.35, color=BLAU, linewidths=0)

ax.set_yticks([0, 1])
ax.set_yticklabels(["gutartig", "bösartig"])
ax.set_xlabel("Tumorgröße (mm)")
ax.set_title("Größere Tumoren sind häufiger bösartig — aber es gibt Ausnahmen")
plt.show()
```

</details>

💬 **Schau dir das Bild einen Moment an, bevor du weiterliest: Was erkennst du?**

<details>
<summary>Antwort aufklappen</summary>

Drei Dinge, und alle drei sind wichtig:

1. **Links unten ist es dicht:** Kleine Tumoren sind meistens gutartig.
2. **Rechts oben ist es dicht:** Große Tumoren sind meistens bösartig.
3. **In der Mitte liegen beide Linien übereinander.** Zwischen ungefähr 15 und 35 mm gibt es
   gutartige *und* bösartige Tumoren derselben Größe.

Punkt 3 ist der entscheidende. Er bedeutet: **Es gibt keine Grenze, die die Daten sauber
trennt.** Man kann sich noch so viel Mühe geben — bei 25 mm liegt man immer manchmal falsch.

Und daraus folgt die eigentliche Einsicht dieses Kapitels: Wenn eine sichere Antwort unmöglich
ist, sollte ein ehrliches Modell auch keine geben. Es sollte sagen, **wie wahrscheinlich**
bösartig ist.

</details>

▶️ Diese Überschneidung ist so zentral, dass wir sie uns noch zweimal anders ansehen: einmal
als Histogramm je Klasse, einmal als Tabelle.

In [ ]:
fig, ax = plt.subplots()

grenzen = np.linspace(0, 60, 25)
ax.hist(df.loc[df["boesartig"] == 0, "tumor_groesse_mm"], bins=grenzen, color=TEAL, alpha=0.75,
        label="gutartig (0)")
ax.hist(df.loc[df["boesartig"] == 1, "tumor_groesse_mm"], bins=grenzen, color=ORANGE, alpha=0.75,
        label="bösartig (1)")

ax.set_xlabel("Tumorgröße (mm)")
ax.set_ylabel("Anzahl Patienten")
ax.set_title("Zwei Klassen, ein breiter Überschneidungsbereich")
ax.legend()
plt.show()

In [ ]:
# ▶️ Anteil bösartiger Tumoren je Größenklasse
klassen = pd.cut(df["tumor_groesse_mm"], bins=[0, 10, 20, 30, 40, 60])

print(f"{'Tumorgröße':<14}{'Patienten':>10}{'davon bösartig':>17}")
print("-" * 41)
for bereich, gruppe in df.groupby(klassen, observed=True):
    beschriftung = f"{bereich.left:.0f}–{bereich.right:.0f} mm"
    print(f"{beschriftung:<14}{len(gruppe):>10}{gruppe['boesartig'].mean():>17.1%}")

📖 Lies die Tabelle von oben nach unten: 0 % → 21 % → 46 % → 76 % → 100 %.

**Der Anteil steigt gleitend an, nicht sprunghaft.** Es gibt keine Größe, ab der es plötzlich
umschlägt. Genau diese gleitende Kurve ist es, die unser Modell gleich nachzeichnen soll.

---
## 3 · Trainings- und Testdaten trennen

📖 Unverändert aus Kapitel 1.2: 80 % Training, 20 % Test. Das Modell lernt nur aus den
Trainingsdaten; die Testdaten sind die Prüfung, deren Fragen es vorher nicht kennt.

▶️ Ausführen. Neu ist nur, dass `y` jetzt Nullen und Einsen enthält statt Preisen.

In [ ]:
zufall = np.random.default_rng(42)
gemischt = zufall.permutation(len(df))
grenze = int(0.8 * len(df))

train = df.iloc[gemischt[:grenze]]
test = df.iloc[gemischt[grenze:]]

x_train = train["tumor_groesse_mm"].to_numpy()
y_train = train["boesartig"].to_numpy().astype(float)
x_test = test["tumor_groesse_mm"].to_numpy()
y_test = test["boesartig"].to_numpy().astype(float)

print(f"Training: {len(x_train)} Patienten, davon {int(y_train.sum())} bösartig")
print(f"Test:     {len(x_test)} Patienten, davon {int(y_test.sum())} bösartig")

---
## 4 · Warum das Modell aus Kapitel 1.2 hier nicht reicht

📖 Wir kennen doch schon ein Modell. Warum nehmen wir nicht einfach wieder die Gerade
$w \cdot x + b$ und lassen sie auf die 0/1-Spalte los?

▶️ Probieren wir es aus — genau so, wie du es in Kapitel 1.2 gemacht hast.

In [ ]:
from sklearn.linear_model import LinearRegression

gerade = LinearRegression().fit(x_train.reshape(-1, 1), y_train)

x_linie = np.linspace(0, 65, 100)

fig, ax = plt.subplots()
ax.scatter(x_train, y_train, s=30, alpha=0.3, color="#94a3b8", linewidths=0)
ax.plot(x_linie, gerade.predict(x_linie.reshape(-1, 1)), color=ORANGE, linewidth=2.5,
        label="lineare Regression")
ax.axhline(0, color=GRAU, linewidth=1, linestyle="--")
ax.axhline(1, color=GRAU, linewidth=1, linestyle="--")

ax.set_xlabel("Tumorgröße (mm)")
ax.set_ylabel("boesartig")
ax.set_title("Eine Gerade passt hier nicht")
ax.legend(loc="upper left")
plt.show()

for groesse in [3.0, 25.0, 60.0, 70.0]:
    print(f"{groesse:>5.0f} mm  →  Vorhersage {gerade.predict([[groesse]])[0]:>6.2f}")

💬 **Das Modell sagt für einen 3-mm-Tumor „−0,17“ und für einen 60-mm-Tumor „1,51“.
Was soll das bedeuten?**

<details>
<summary>Antwort aufklappen</summary>

Nichts. Es gibt keine Klasse −0,17, und eine Wahrscheinlichkeit von 151 % existiert nicht.

Das ist kein Schönheitsfehler, sondern grundsätzlich: **Eine Gerade hört nicht auf.** Sie läuft
in beide Richtungen unbegrenzt weiter. Egal wie man $w$ und $b$ wählt — irgendwo verlässt sie
den Bereich zwischen 0 und 1.

</details>

📖 Was wir bräuchten, ist eine Ausgabe, die **niemals kleiner als 0 und niemals größer als 1**
wird. Dann könnten wir sie als **Wahrscheinlichkeit** lesen: *„Dieser Tumor ist mit 80 %
Wahrscheinlichkeit bösartig.“*

Die Lösung ist verblüffend einfach — und sie ist der ganze Unterschied zwischen linearer und
logistischer Regression:

> **Wir behalten die Gerade und bilden ihr Ergebnis anschließend zwischen 0 und 1.**

Das Modell bekommt also einen zweiten Schritt:

$$z = w \cdot x + b \qquad\text{(die Gerade aus Kapitel 1.2 — unverändert)}$$

$$p = \sigma(z) = \frac{1}{1 + e^{-z}} \qquad\text{(neu: die Sigmoid-Funktion)}$$

Das $z$ ist eine beliebige Zahl — der **Score**. Das $p$ ist die Wahrscheinlichkeit für
„bösartig“. Die Funktion $\sigma$ (sprich: *Sigma*) heißt **Sigmoid-Funktion**, weil ihr Graph
wie ein liegendes S aussieht.

### 🛠️ Challenge 3 — Die Sigmoid-Funktion

Schreibe die Funktion, die aus einer beliebigen Zahl $z$ einen Wert zwischen 0 und 1 macht:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

*Tipp: `np.exp(...)` ist die e-Funktion. Mehr als eine Zeile brauchst du nicht.*

In [ ]:
def sigmoid(z):
    """Bildet jede Zahl auf einen Wert zwischen 0 und 1."""
    # TODO: Ersetze die nächste Zeile durch die Formel
    raise NotImplementedError("Challenge 3: sigmoid() implementieren")

In [ ]:
# ✅ Selbsttest
assert sigmoid(0) == 0.5
assert sigmoid(10) > 0.99
assert sigmoid(-10) < 0.01
assert np.isclose(sigmoid(2) + sigmoid(-2), 1.0)
assert np.allclose(sigmoid(np.array([0.0, 100.0])), [0.5, 1.0])
print("✅ Challenge 3 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def sigmoid(z):
    """Bildet jede Zahl auf einen Wert zwischen 0 und 1."""
    return 1 / (1 + np.exp(-z))
```

Drei Eigenschaften, die gleich wichtig werden: Bei $z = 0$ kommt genau **0,5** heraus. Für große positive $z$ nähert sich das Ergebnis der **1**, für große negative der **0** — erreicht wird beides nie. Und sie ist symmetrisch: $\sigma(-z) = 1 - \sigma(z)$.

</details>

▶️ So sieht sie aus. Egal was du hineinsteckst — es kommt nie etwas unter 0 oder über 1 heraus.

In [ ]:
z = np.linspace(-8, 8, 300)

fig, ax = plt.subplots()
ax.plot(z, sigmoid(z), color=BLAU, linewidth=2.5)
ax.axhline(0.5, color=GRAU, linewidth=1, linestyle="--")
ax.axvline(0, color=GRAU, linewidth=1, linestyle="--")
ax.scatter([0], [0.5], color=ORANGE, s=60, zorder=3)
ax.annotate("z = 0  →  p = 0,5\n(maximal unsicher)", xy=(0, 0.5), xytext=(1.2, 0.28),
            color=ORANGE, arrowprops=dict(arrowstyle="->", color=ORANGE))

ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("z  (die Zahl, die aus der Geraden kommt)")
ax.set_ylabel("p  (Wahrscheinlichkeit)")
ax.set_title("Die Sigmoid-Funktion: aus jeder Zahl wird eine Wahrscheinlichkeit")
plt.show()

### 🛠️ Challenge 4 — Das Modell

Jetzt setzt du beide Schritte zusammen: erst die Gerade, dann die Sigmoid-Funktion.

Das ist **das komplette Modell der logistischen Regression** — eine Zeile.

*Tipp: In Kapitel 1.2 hast du `w * x + b` zurückgegeben. Diesmal gibst du dasselbe zurück,
nur vorher durch deine `sigmoid()` geschickt.*

In [ ]:
def wahrscheinlichkeit(x, w, b):
    """Wie wahrscheinlich ist es, dass ein Tumor der Größe x bösartig ist?"""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Challenge 4: wahrscheinlichkeit() implementieren")

In [ ]:
# ✅ Selbsttest
assert np.isclose(wahrscheinlichkeit(25, 0.1, -2.5), 0.5)
assert wahrscheinlichkeit(60, 0.1, -2.5) > 0.95
assert wahrscheinlichkeit(0, 0.1, -2.5) < 0.10
assert np.allclose(wahrscheinlichkeit(np.array([25.0, 60.0]), 0.1, -2.5), [0.5, 0.971], atol=1e-3)
print("✅ Challenge 4 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def wahrscheinlichkeit(x, w, b):
    """Wie wahrscheinlich ist es, dass ein Tumor der Größe x bösartig ist?"""
    return sigmoid(w * x + b)
```

Mehr ist es nicht. Die logistische Regression ist die lineare Regression mit einer Sigmoid-Funktion obendrauf — deshalb heißt sie auch „Regression“, obwohl sie klassifiziert.

</details>

▶️ Wie in Kapitel 1.2 raten wir wieder drei Modelle und schauen, welches am besten passt.
Diesmal sind es keine Geraden, sondern S-Kurven.

In [ ]:
kandidaten = [(0.08, -2.0), (0.15, -4.5), (0.40, -8.0)]
farben = [GRAU, BLAU, ORANGE]

fig, ax = plt.subplots()
ax.scatter(x_train, y_train, s=30, alpha=0.25, color="#94a3b8", linewidths=0)

x_linie = np.linspace(0, 65, 300)
for (w, b), farbe in zip(kandidaten, farben):
    ax.plot(x_linie, wahrscheinlichkeit(x_linie, w, b), color=farbe, linewidth=2.5,
            label=f"w = {w}, b = {b}")

ax.axhline(0.5, color=GRAU, linewidth=1, linestyle="--")
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("Tumorgröße (mm)")
ax.set_ylabel("p (bösartig)")
ax.set_title("Welche S-Kurve ist die beste?")
ax.legend(loc="upper left")
plt.show()

💬 Was machen $w$ und $b$ hier eigentlich?

<details>
<summary>Antwort aufklappen</summary>

* $w$ bestimmt, **wie steil** die Kurve ist. Großes $w$ = das Modell wechselt schnell von „nein“
  zu „ja“, es ist sich schnell sicher. Kleines $w$ = ein langer, unsicherer Übergangsbereich.
* $b$ verschiebt die Kurve **nach links oder rechts** — also *bei welcher Tumorgröße* der
  Umschwung passiert.

Die Stelle, an der die Kurve die 0,5 kreuzt, ist genau dort, wo $z = w \cdot x + b = 0$ ist,
also bei $x = -b / w$. Für die orange Kurve: $8{,}0 / 0{,}4 = 20$ mm.

</details>

Und wieder dieselbe Frage wie in Kapitel 1.2: **Woher wollen wir wissen, welche Kurve die beste
ist?** Wir brauchen eine Zahl.

---
## 5 · Die Kostenfunktion

📖 Nach dem Modell müssen wir auch die **Kostenfunktion** austauschen. In Kapitel 1.2 war das
der MSE. Bei der logistischen Regression nimmt man stattdessen den **Log-Loss**:

$$\text{Log-Loss} = -\frac{1}{n}\sum_{i=1}^{n}
\Big[\, y_i \log(p_i) + (1 - y_i)\log(1 - p_i) \,\Big]$$

▶️ So sieht diese Strafe aus:

In [ ]:
p = np.linspace(0.001, 0.999, 300)

fig, ax = plt.subplots()
ax.plot(p, -np.log(p), color=ORANGE, linewidth=2.5, label="Tumor ist bösartig (y = 1)")
ax.plot(p, -np.log(1 - p), color=TEAL, linewidth=2.5, label="Tumor ist gutartig (y = 0)")

ax.set_ylim(0, 5)
ax.set_xlabel("p — die Wahrscheinlichkeit, die das Modell für 'bösartig' ausgibt")
ax.set_ylabel("Strafe")
ax.set_title("Sicher und falsch wird teuer bestraft")
ax.legend()
plt.show()

📖 Lies die orange Kurve (der Tumor **ist** bösartig): Sagt das Modell 0,9, ist die Strafe fast
null. Sagt es 0,01 — sicher, und sicher falsch — schießt sie steil nach oben.

Und wie beim MSE wird am Ende über alle Patienten gemittelt. **Trainieren heißt: diese Zahl so
klein wie möglich machen.**

### 🛠️ Challenge 5 — Log-Loss

Schreibe die Kostenfunktion.

*Tipp: Wieder keine Schleife nötig. `np.log(...)` rechnet elementweise, `np.mean(...)` mittelt.
Achte auf das Minuszeichen ganz vorn.*

In [ ]:
def log_loss(y_wahr, p):
    """Mittlerer Log-Loss zwischen echten Labels (0/1) und vorhergesagten Wahrscheinlichkeiten."""
    # TODO: Ersetze die nächste Zeile durch die Formel
    raise NotImplementedError("Challenge 5: log_loss() implementieren")

In [ ]:
# ✅ Selbsttest
assert np.isclose(log_loss(np.array([1.0, 0.0]), np.array([0.5, 0.5])), 0.6931, atol=1e-4)
assert np.isclose(log_loss(np.array([1.0]), np.array([0.9])), 0.1054, atol=1e-4)
assert np.isclose(log_loss(np.array([0.0]), np.array([0.1])), 0.1054, atol=1e-4)
assert log_loss(np.array([1.0]), np.array([0.99])) < log_loss(np.array([1.0]), np.array([0.70]))
print("✅ Challenge 5 gelöst")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def log_loss(y_wahr, p):
    """Mittlerer Log-Loss zwischen echten Labels (0/1) und vorhergesagten Wahrscheinlichkeiten."""
    return float(-np.mean(y_wahr * np.log(p) + (1 - y_wahr) * np.log(1 - p)))
```

Die zwei Summanden sehen komplizierter aus, als sie sind: Ist $y = 1$, fällt der hintere weg ($1 - y = 0$); ist $y = 0$, fällt der vordere weg. Übrig bleibt immer genau die Strafe für die richtige Antwort.

</details>

In [ ]:
# ▶️ Was sagt der Log-Loss über unsere drei geratenen Kurven?
for w, b in kandidaten:
    kosten = log_loss(y_train, wahrscheinlichkeit(x_train, w, b))
    print(f"w = {w:<6} b = {b:<6}  →  Log-Loss = {kosten:.4f}")

▶️ Und wie in Kapitel 1.2 probieren wir nicht drei, sondern 200 Werte für $w$ durch. ($b$ halten
wir dabei bei −4,5 fest, sonst bräuchten wir ein 3D-Bild.)

In [ ]:
w_werte = np.linspace(0.0, 0.5, 200)
kosten_werte = [log_loss(y_train, wahrscheinlichkeit(x_train, w, -4.5)) for w in w_werte]
bestes_w = w_werte[int(np.argmin(kosten_werte))]

fig, ax = plt.subplots()
ax.plot(w_werte, kosten_werte, color=BLAU, linewidth=2)
ax.scatter([bestes_w], [min(kosten_werte)], color=ORANGE, s=60, zorder=3)
ax.annotate(f"bestes w ≈ {bestes_w:.3f}",
            xy=(bestes_w, min(kosten_werte)),
            xytext=(bestes_w + 0.07, min(kosten_werte) + 0.5),
            color=ORANGE, arrowprops=dict(arrowstyle="->", color=ORANGE))

ax.set_xlabel("w")
ax.set_ylabel("Log-Loss")
ax.set_title("Wieder eine Schüssel (b = −4,5 festgehalten)")
plt.show()

📖 **Dieselbe Schüssel wie in Kapitel 1.2** — nur mit einer anderen Kostenfunktion. Und die
Aufgabe ist wieder dieselbe: den tiefsten Punkt finden.

Also auch dasselbe Verfahren: **Gradientenabstieg**. Im Nebel bergab, Schritt für Schritt.

Ein Unterschied ist allerdings wichtig: Für die lineare Regression gibt es zusätzlich eine
exakte Formel, die das Minimum in einem Rutsch ausrechnet. **Für die logistische Regression gibt
es die nicht.** Hier bleibt nur das schrittweise Abtasten — dasselbe Verfahren, mit dem auch
neuronale Netze und LLMs trainiert werden. Ab hier ist Gradientenabstieg nicht mehr die
elegante Variante, sondern die einzige.

▶️ Schreiben müssen wir ihn trotzdem nicht selbst — `scikit-learn` übernimmt das.

---
## 6 · Trainieren

In [ ]:
from sklearn.linear_model import LogisticRegression

modell = LogisticRegression()
modell.fit(x_train.reshape(-1, 1), y_train)

w_trainiert = modell.coef_[0][0]
b_trainiert = modell.intercept_[0]

print(f"w = {w_trainiert:.4f}")
print(f"b = {b_trainiert:.4f}")
print()
print(f"Dein Modell: p(bösartig) = sigmoid({w_trainiert:.4f} · Tumorgröße {b_trainiert:+.4f})")

In [ ]:
# ▶️ Geraten gegen trainiert
print(f"{'Modell':<32}{'Log-Loss (Trainingsdaten)':>28}")
print("-" * 60)
for w, b in kandidaten:
    beschriftung = f"geraten: w={w}, b={b}"
    print(f"{beschriftung:<32}{log_loss(y_train, wahrscheinlichkeit(x_train, w, b)):>28.4f}")
print(f"{'scikit-learn':<32}"
      f"{log_loss(y_train, wahrscheinlichkeit(x_train, w_trainiert, b_trainiert)):>28.4f}")

In [ ]:
# ▶️ Dein trainiertes Modell
x_linie = np.linspace(0, 65, 300)
grenz_groesse = -b_trainiert / w_trainiert

fig, ax = plt.subplots()
ax.scatter(x_train, y_train, s=30, alpha=0.3, color="#94a3b8", linewidths=0)
ax.plot(x_linie, wahrscheinlichkeit(x_linie, w_trainiert, b_trainiert),
        color=ORANGE, linewidth=2.5, label="trainiertes Modell")
ax.axhline(0.5, color=GRAU, linewidth=1, linestyle="--")
ax.axvline(grenz_groesse, color=TEAL, linewidth=2, linestyle=":")
ax.text(grenz_groesse + 1.5, 0.06, f"{grenz_groesse:.1f} mm\np = 0,5", color=TEAL)

ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("Tumorgröße (mm)")
ax.set_ylabel("p (bösartig)")
ax.set_title("Dein trainiertes Modell")
ax.legend(loc="upper left")
plt.show()

📖 **Was die beiden Zahlen bedeuten.**

Die Kurve kreuzt die 0,5 bei rund **25,8 mm**. Das ist die **Entscheidungsgrenze**: Bis dahin
hält das Modell „gutartig“ für wahrscheinlicher, danach „bösartig“. Sie liegt bei $x = -b/w$ —
genau dort, wo die Gerade $z = w \cdot x + b$ durch null geht.

$w$ ist positiv: mehr Millimeter → höherer Score → höhere Wahrscheinlichkeit. Der Wert wirkt
klein (0,18), weil er sich auf **einen** Millimeter bezieht. 10 mm mehr heben den Score $z$ um
1,8 — und das ist auf der Sigmoid-Skala ein großer Sprung.

Beachte, was dabei **nicht** passiert ist: Niemand hat dem Modell gesagt, wo die Grenze liegt
oder wie steil die Kurve sein soll. Es hat beide Zahlen allein aus 240 Ja/Nein-Antworten
gezogen. Genau das meint „Lernen aus Daten“.

<details>
<summary>💡 Für alle, die es genauer wollen: was 0,18 exakt bedeutet</summary>

$z$ ist kein bedeutungsloses Zwischenergebnis — es sind die **Log-Odds**, also der Logarithmus
des Chancenverhältnisses $\frac{p}{1-p}$. Ein $w$ von 0,1775 heißt: Jeder zusätzliche
Millimeter multipliziert die *Chance* auf „bösartig“ mit $e^{0{,}1775} \approx 1{,}19$, also
+19 %. Über 10 mm gerechnet: $e^{1{,}775} \approx 5{,}9$ — die Chance versechsfacht sich.

Deshalb ist die logistische Regression in Medizin und Versicherung so beliebt: Die gelernten
Gewichte lassen sich als „Risiko mal soundsoviel“ vorlesen.

</details>

---
## 7 · Von der Wahrscheinlichkeit zur Entscheidung

📖 Bis hierher liefert das Modell eine Wahrscheinlichkeit — 0,73 zum Beispiel. Irgendwann muss
daraus aber eine Entscheidung werden: weiter beobachten oder Gewebeprobe entnehmen?

Dafür braucht es einen **Schwellenwert** (*threshold*): Ab welcher Wahrscheinlichkeit sagen wir
„bösartig“? Der Standardwert ist 0,5, aber das ist **eine Konvention, keine Naturkonstante**.

Halte die beiden Schritte sauber auseinander — das ist der Kern dieses Abschnitts:

| | |
|---|---|
| **Regressionsteil** | Tumorgröße → 0,82 |
| **Klassifikationsteil** | 0,82 → Schwelle 0,5 → **bösartig** |

Das Modell macht nur den ersten Schritt. Der zweite ist eine Entscheidung, die **wir** treffen.

▶️ Erst einmal mit 0,5. Wie oft liegt das Modell auf den **Testdaten** richtig?

In [ ]:
def klassifiziere(p, schwelle=0.5):
    """Macht aus Wahrscheinlichkeiten harte 0/1-Entscheidungen."""
    return (p >= schwelle).astype(int)


p_test = wahrscheinlichkeit(x_test, w_trainiert, b_trainiert)
vorhersage_test = klassifiziere(p_test, 0.5)

treffer = (vorhersage_test == y_test).mean()
immer_gutartig = (y_test == 0).mean()

print(f"Treffergenauigkeit des Modells:  {treffer:.1%}")
print(f"Immer stur 'gutartig' antworten: {immer_gutartig:.1%}")

📖 76,7 % richtig — gegenüber 55,0 %, die man schon bekommt, wenn man **immer** „gutartig“ sagt.
Das Modell hat also etwas gelernt. Perfekt wird es nie: Erinnerst du dich an den
Überschneidungsbereich aus Abschnitt 2? Der lässt sich nicht wegrechnen.

**Diese eine Prozentzahl verschweigt aber das Wichtigste: *welche* Fehler das Modell macht.**
Es gibt nämlich zwei verschiedene, und hier sind sie ganz sicher nicht gleich schlimm. Deshalb
schaut man sich alle vier Fälle einzeln an — das ist die **Konfusionsmatrix**.

In [ ]:
def konfusionsmatrix(y_wahr, y_vorhersage):
    """Zählt die vier möglichen Fälle."""
    richtig_ja = int(((y_vorhersage == 1) & (y_wahr == 1)).sum())
    falsch_ja = int(((y_vorhersage == 1) & (y_wahr == 0)).sum())
    falsch_nein = int(((y_vorhersage == 0) & (y_wahr == 1)).sum())
    richtig_nein = int(((y_vorhersage == 0) & (y_wahr == 0)).sum())
    return richtig_ja, falsch_ja, falsch_nein, richtig_nein


richtig_ja, falsch_ja, falsch_nein, richtig_nein = konfusionsmatrix(y_test, vorhersage_test)

print("                          Modell sagt:")
print("                     bösartig     gutartig")
print(f"tatsächlich bösartig    {richtig_ja:>5}        {falsch_nein:>5}   ← ÜBERSEHEN")
print(f"tatsächlich gutartig    {falsch_ja:>5}        {richtig_nein:>5}")
print("                          ↑")
print("                     Fehlalarm")

📖 Zwei Sorten Fehler — und in der Medizin sind sie himmelweit voneinander entfernt:

* **Fehlalarm:** Das Modell sagt bösartig, der Tumor ist gutartig. Folge: eine unnötige
  Gewebeprobe, Angst, Kosten. Unangenehm, aber reparabel.
* **Übersehen:** Das Modell sagt gutartig, der Tumor ist bösartig. Folge: Der Krebs bleibt
  unbehandelt. **Das ist nicht reparabel.**

Bei 7 übersehenen von 27 bösartigen Tumoren würde niemand dieses Modell einsetzen wollen.

**Und genau daran dreht der Schwellenwert.** Er ändert nicht, was das Modell weiß — die
Wahrscheinlichkeiten bleiben exakt dieselben. Er ändert nur, ab wann wir „bösartig“ sagen, und
damit, welche der beiden Fehlerarten wir häufiger in Kauf nehmen.

▶️ Schau dir an, wie sich die Zahlen verschieben:

In [ ]:
print(f"{'Schwelle':>9}{'Fehlalarme':>13}{'ÜBERSEHEN':>12}{'Treffer':>10}"
      f"{'gefundene bösartige':>22}")
print("-" * 66)
for schwelle in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    vorhersage = klassifiziere(p_test, schwelle)
    r_ja, f_ja, f_nein, r_nein = konfusionsmatrix(y_test, vorhersage)
    gefunden = r_ja / (r_ja + f_nein)
    print(f"{schwelle:>9.1f}{f_ja:>13}{f_nein:>12}"
          f"{(vorhersage == y_test).mean():>10.1%}{gefunden:>22.1%}")

💬 **Welchen Schwellenwert würdest du für ein Krebs-Screening wählen — und warum ist die
Treffergenauigkeit dabei fast egal?**

<details>
<summary>Antwort aufklappen</summary>

Einen **niedrigen**, etwa 0,2. Dann übersieht das Modell nur noch 2 statt 7 bösartige Tumoren
und findet 92,6 % von ihnen. Der Preis: 18 statt 7 Fehlalarme — und die Treffergenauigkeit
**fällt** von 76,7 % auf 66,7 %.

Das ist der Punkt: **Die schlechtere Treffergenauigkeit ist hier das bessere Modell.** Wer nach
der höchsten Prozentzahl optimiert, optimiert an der Wirklichkeit vorbei. 11 zusätzliche
unnötige Gewebeproben gegen 5 zusätzlich entdeckte Krebserkrankungen — das ist kein
knapper Fall.

**Die Lehre:** Der beste Schwellenwert steht nicht in den Daten. Er ergibt sich daraus, was ein
Fehler jeweils kostet — und das ist eine Frage an die Fachwelt, nicht an das Modell. Genau
deshalb gibt ein gut gebautes Modell **Wahrscheinlichkeiten** aus und überlässt die Schwelle
denen, die die Konsequenzen tragen.

Übrigens dieselbe Überlegung mit umgekehrtem Vorzeichen beim Spamfilter: Dort ist der Fehlalarm
(wichtige Mail im Spam-Ordner) schlimmer als das Übersehen (eine Spam-Mail im Posteingang).
Dort will man eine **hohe** Schwelle.

</details>

---
## 8 · Dein Modell im Einsatz

📖 Wie in Kapitel 1.2 besteht dein trainiertes Modell aus genau **zwei Zahlen**: `w_trainiert`
und `b_trainiert`. Der Unterschied: Was herauskommt, ist jetzt eine Wahrscheinlichkeit.

### 🛠️ Challenge 6 — Eine neue Patientin

📬 In der Sprechstunde:

> Bei einer Patientin wurde ein Tumor von **22 mm** festgestellt.
> Was sagt dein Modell?

Rechne die Wahrscheinlichkeit auf beide Arten aus:

1. mit deiner eigenen `wahrscheinlichkeit()`-Funktion aus Challenge 4,
2. mit `modell.predict_proba(...)` von `scikit-learn`.

*Tipp zu 2: `predict_proba` gibt für jeden Fall **zwei** Zahlen zurück — die Wahrscheinlichkeit
für Klasse 0 und die für Klasse 1. Du willst die zweite:
`modell.predict_proba([[22.0]])[0][1]`.*

In [ ]:
tumor_neu = 22.0

# TODO 1: mit deiner eigenen Funktion
p_eigene_funktion = ...

# TODO 2: mit scikit-learn
p_sklearn = ...

print(f"Deine Funktion: {p_eigene_funktion:.1%}")
print(f"scikit-learn:   {p_sklearn:.1%}")

In [ ]:
# ✅ Selbsttest
assert abs(p_eigene_funktion - 0.338) < 0.01, "Erwartet werden rund 33,8 %"
assert np.isclose(p_eigene_funktion, p_sklearn), "Beide Wege müssen dasselbe liefern"
print(f"✅ Challenge 6 gelöst — {p_eigene_funktion:.1%} Wahrscheinlichkeit für 'bösartig'.")
print(f"   Entscheidung bei Schwelle 0,5: "
      f"{'bösartig' if p_eigene_funktion >= 0.5 else 'gutartig'}")
print(f"   Entscheidung bei Schwelle 0,2: "
      f"{'bösartig' if p_eigene_funktion >= 0.2 else 'gutartig'}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
tumor_neu = 22.0

p_eigene_funktion = wahrscheinlichkeit(tumor_neu, w_trainiert, b_trainiert)

p_sklearn = modell.predict_proba([[tumor_neu]])[0][1]

print(f"Deine Funktion: {p_eigene_funktion:.1%}")
print(f"scikit-learn:   {p_sklearn:.1%}")
```

`predict_proba()` macht nichts anderes als deine eine Zeile `sigmoid(w * x + b)` — auf die Nachkommastelle dasselbe Ergebnis.

</details>

💬 **33,8 % — unter 0,5. Ist der Tumor also gutartig?**

<details>
<summary>Antwort aufklappen</summary>

Das steht da nicht. Da steht: **Etwa jeder dritte Tumor dieser Größe ist bösartig.**

„Gutartig“ wäre die Antwort *einer bestimmten Schwelle*, nämlich 0,5 — und die haben wir im
vorigen Abschnitt gerade als ungeeignet für diese Frage verworfen. Bei der Schwelle 0,2, die
für ein Screening sinnvoller ist, lautet dieselbe Zahl: **bösartig, bitte abklären.**

Ein Drittel Risiko ist in der Medizin nichts, was man wegschickt. Die Wahrscheinlichkeit ist die
eigentliche Antwort des Modells — die Klasse ist nur eine Vereinfachung davon, und man sollte
sie erst ganz zum Schluss bilden.

</details>

In [ ]:
# ▶️ Zum Spielen: eigene Tumorgrößen einsetzen und ausführen
for groesse in [5, 10, 15, 20, 25, 30, 40, 50]:
    p = wahrscheinlichkeit(float(groesse), w_trainiert, b_trainiert)
    balken = "█" * int(round(p * 30))
    print(f"{groesse:>3} mm  →  {p:>6.1%}  {balken}")

💬 Das Modell gibt einem 50-mm-Tumor 98,7 %. **Heißt das, es ist sich fast sicher — oder heißt
es nur, dass es keine Ahnung hat, was es nicht weiß?**

<details>
<summary>Antwort aufklappen</summary>

Letzteres. Das Modell hat nie etwas anderes gesehen als **eine Zahl in Millimetern**. Form,
Dichte, Lage, Wachstumsgeschwindigkeit, Alter, Vorgeschichte — davon stand nichts in `x_train`.
Zwei Patienten mit 50 mm bekommen exakt dieselbe Antwort.

Die 98,7 % beschreiben, wie sicher sich das Modell **innerhalb seiner winzigen Welt** ist, nicht
wie sicher die Aussage über die Wirklichkeit ist. Das ist eine der wichtigsten Unterscheidungen
im ganzen Feld — und sie gilt genauso für jedes LLM, das dir eine Antwort in sehr
selbstbewusstem Tonfall präsentiert.

</details>

---
# 🎉 Fertig!

Du hast eine Klassifikation gebaut. Und wenn du zurückblätterst, siehst du: **Fast nichts war
neu.**

| | Kapitel 1.2 · Linear | Kapitel 1.3 · Logistisch |
|---|---|---|
| Frage | Wie teuer? (eine Zahl) | Bösartig, ja/nein? (eine Klasse) |
| Modell | $w \cdot x + b$ | $\sigma(w \cdot x + b)$ |
| Ausgabe | ein Preis | eine Wahrscheinlichkeit |
| Kostenfunktion | MSE | Log-Loss |
| Training | Gradientenabstieg | Gradientenabstieg |
| Danach | fertig | + Schwellenwert wählen |

**Zwei ausgetauschte Bausteine.** Das ist der Punkt, den du mitnehmen solltest: Machine Learning
ist nicht für jede Aufgabe ein neues Verfahren, sondern meistens dasselbe Gerüst mit einem
anderen Teil darin.

Und das ist der Weg, den dein Modell gerade genommen hat:

```
Tumorgröße
    ↓
w · x + b          ← gewichtete Berechnung
    ↓
Sigmoid
    ↓
Wahrscheinlichkeit
    ↓
Schwellenwert
    ↓
0 oder 1
```

Du hast gerade eines gebaut.

### ☕ Bis dahin: Kaffee
Und wenn du magst: schau mal nach links und rechts. Jemandem beim Debuggen zu helfen ist die
beste Art, selbst zu merken, ob man es verstanden hat.